# Домашняя работа 3. Логистическая регрессия: синтетические и реальные данные

**Курс:** Машинное обучение и анализ данных

| Часть | Содержание |
|---|---|
| 0 | Импорты, настройки, вспомогательные функции |
| I (3.1–3.4) | Синтетика: набор «Инь-Ян», полиномиальные признаки, L2/L1-регуляризация, артефакты модели (веса, метрики, границы решений), выводы |
| II (4.1–4.5) | Реальные данные **Adult Census Income** (OpenML): EDA, предобработка, baseline, улучшение качества (категориальные признаки, свои признаки, полиномы, веса классов, `SGDClassifier`, порог), анализ ошибок, выводы |

**Принципы**
- все обучаемые преобразования живут внутри `Pipeline` и обучаются только на train — утечки данных нет;
- модель выбирается по кросс-валидации на train, тест используется один раз для итоговой оценки;
- метрики подбираются под задачу: на несбалансированных реальных данных одной accuracy недостаточно;
- каждый рисунок подписан и сопровождается комментарием.

## 0. Импорты и настройки

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from IPython.display import display

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.datasets import fetch_openml
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             average_precision_score, log_loss, confusion_matrix, ConfusionMatrixDisplay,
                             RocCurveDisplay, PrecisionRecallDisplay)
from sklearn.model_selection import (train_test_split, StratifiedKFold, cross_validate, cross_val_predict, GridSearchCV)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler, OneHotEncoder

warnings.filterwarnings("ignore")   # предупреждения о сходимости / депрекейты параметров не влияют на выводы
SEED = 42
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .3, "axes.titlesize": 12,
                     "axes.labelsize": 10, "legend.fontsize": 9})

In [ ]:
def clf_metrics(y_true, proba, thr=0.5):
    # набор метрик классификации (по вероятностям и порогу thr)
    pred = (proba >= thr).astype(int)
    return {"Accuracy": accuracy_score(y_true, pred), "Precision": precision_score(y_true, pred, zero_division=0),
            "Recall": recall_score(y_true, pred), "F1": f1_score(y_true, pred),
            "ROC-AUC": roc_auc_score(y_true, proba), "PR-AUC": average_precision_score(y_true, proba),
            "LogLoss": log_loss(y_true, np.clip(proba, 1e-9, 1 - 1e-9))}

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

# Часть I. Синтетические данные (3.1–3.4)

## 3.1 Генерация данных: «Инь-Ян»

Классическая задача с нелинейной границей: два класса «закручены» в единичном круге (большие «головы» и маленькие «глазки» противоположного класса). Генератор написан вручную: точки равномерно в круге, класс определяется геометрией, затем добавляется гауссов шум к координатам (выборка 500 точек, `train/val = 70/30`, стратифицированно).

In [ ]:
def make_yin_yang(n=500, noise=0.05, seed=SEED):
    rng = np.random.default_rng(seed)
    pts = []
    while len(pts) < n:                      # равномерно в единичном круге (rejection sampling)
        p = rng.uniform(-1, 1, 2)
        if p @ p <= 1:
            pts.append(p)
    P = np.array(pts)
    x, y = P[:, 0], P[:, 1]
    d_up, d_low = np.hypot(x, y - .5), np.hypot(x, y + .5)
    cls = (x < 0).astype(int)                # две половины круга
    cls[d_up < .5] = 0;  cls[d_low < .5] = 1  # две «головы»
    cls[d_up < .15] = 1; cls[d_low < .15] = 0  # две «точки-глаза»
    return P + rng.normal(0, noise, P.shape), cls

Xs, ys = make_yin_yang()
Xs_tr, Xs_val, ys_tr, ys_val = train_test_split(Xs, ys, test_size=.3, random_state=SEED, stratify=ys)

fig, ax = plt.subplots(figsize=(5.2, 5))
for c, col in [(0, "tab:blue"), (1, "tab:red")]:
    ax.scatter(*Xs_tr[ys_tr == c].T, s=18, c=col, label=f"класс {c}, train")
    ax.scatter(*Xs_val[ys_val == c].T, s=30, c=col, marker="x", label=f"класс {c}, val")
ax.set(title=f"Инь-Ян: train={len(ys_tr)}, val={len(ys_val)}", xlabel="x1", ylabel="x2", aspect="equal"); ax.legend(loc="upper right", fontsize=7)
plt.show()

*Комментарий.* Классы не разделяются никакой прямой: граница «S-образная» с вложенными областями. Это идеальный полигон для демонстрации того, что **логистическая регрессия линейна по признакам**, а нелинейность нужно подавать через полиномы.

## 3.2 Логистическая регрессия и полиномы

### 3.2.1 Подбор степени полинома

Пайплайн: `PolynomialFeatures(d) → StandardScaler → LogisticRegression(C=1000)` (слабая регуляризация — чтобы увидеть именно эффект степени). Смотрим accuracy и **log-loss** на train/val: accuracy грубая, log-loss чувствителен к уверенности предсказаний и первым показывает переобучение.

In [ ]:
def poly_clf(degree, C=1000.0, penalty="l2"):
    solver = "liblinear" if penalty == "l1" else "lbfgs"
    return Pipeline([("poly", PolynomialFeatures(degree, include_bias=False)), ("scale", StandardScaler()),
                     ("model", LogisticRegression(C=C, penalty=penalty, solver=solver, max_iter=20000))])

def plot_boundary(model, X, y, ax, title, lim=1.3):
    xx, yy = np.meshgrid(np.linspace(-lim, lim, 220), np.linspace(-lim, lim, 220))
    p = model.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:, 1].reshape(xx.shape)
    cs = ax.contourf(xx, yy, p, levels=np.linspace(0, 1, 11), cmap="RdBu_r", alpha=.75)
    ax.contour(xx, yy, p, levels=[.5], colors="k", linewidths=1.6)
    ax.scatter(*X.T, c=y, cmap="RdBu_r", edgecolor="k", s=14, linewidth=.3)
    ax.set(title=title, xlim=(-lim, lim), ylim=(-lim, lim), aspect="equal")
    return cs

rows = []
for d in range(1, 16):
    m = poly_clf(d).fit(Xs_tr, ys_tr)
    ptr, pva = m.predict_proba(Xs_tr)[:, 1], m.predict_proba(Xs_val)[:, 1]
    rows.append({"degree": d, "train_acc": accuracy_score(ys_tr, ptr > .5), "val_acc": accuracy_score(ys_val, pva > .5),
                 "train_logloss": log_loss(ys_tr, ptr), "val_logloss": log_loss(ys_val, pva), "n_features": m.named_steps["model"].coef_.size})
deg_df = pd.DataFrame(rows)
best_deg = int(deg_df.loc[deg_df["val_logloss"].idxmin(), "degree"])
HIGH_DEG = 15

fig, axes = plt.subplots(1, 2, figsize=(13, 4.3))
axes[0].plot(deg_df.degree, deg_df.train_acc, "o-", label="train"); axes[0].plot(deg_df.degree, deg_df.val_acc, "s-", label="val")
axes[0].axvline(best_deg, color="green", ls=":", label=f"min val log-loss: степень {best_deg}")
axes[0].set(title="Accuracy vs степень полинома", xlabel="степень", ylabel="accuracy"); axes[0].legend()
axes[1].plot(deg_df.degree, deg_df.train_logloss, "o-", label="train"); axes[1].plot(deg_df.degree, deg_df.val_logloss, "s-", label="val")
axes[1].axvline(best_deg, color="green", ls=":")
axes[1].set(title="Log-loss vs степень полинома", xlabel="степень", ylabel="log-loss"); axes[1].legend()
plt.show()

fig, axes = plt.subplots(1, 4, figsize=(18, 4.6))
shown = sorted({1, best_deg, 7, HIGH_DEG}) if best_deg not in (1, 7) else [1, 3, 7, HIGH_DEG]
for ax, d in zip(axes, shown):
    m = poly_clf(d).fit(Xs_tr, ys_tr)
    va = accuracy_score(ys_val, m.predict(Xs_val))
    cs = plot_boundary(m, Xs_tr, ys_tr, ax, f"степень {d}, val acc = {va:.3f}")
fig.colorbar(cs, ax=axes, label="P(класс 1)", shrink=.8); plt.show()
display(deg_df.round(3).set_index("degree").T.iloc[:, [0, 1, 2, 4, 6, 8, 10, 12, 14]])

*Комментарий.* Степень 1 даёт почти прямую границу и сильное **недообучение**. С ростом степени accuracy растёт и быстро выходит на плато, а вот **val log-loss проходит минимум и затем растёт**: модель становится излишне уверенной на «шумовых» участках (см. рваные границы при высокой степени). То есть переобучение видно по log-loss раньше и отчётливее, чем по accuracy.

### 3.2.2 Регуляризация: L2 (Ridge-подобная) и L1 (Lasso-подобная)

Фиксируем **избыточную степень (15)** — 135 признаков на 350 точек — и меняем силу регуляризации `C` (обратная величина: **меньше C — сильнее регуляризация**). L1 дополнительно зануляет веса.

In [ ]:
C_l2 = np.logspace(-2, 3, 11)
C_l1 = np.logspace(-2, 1.3, 8)
def c_sweep(pen, Cs):
    out = []
    for C in Cs:
        m = poly_clf(HIGH_DEG, C=C, penalty=pen).fit(Xs_tr, ys_tr)
        ptr, pva = m.predict_proba(Xs_tr)[:, 1], m.predict_proba(Xs_val)[:, 1]
        out.append((log_loss(ys_tr, ptr), log_loss(ys_val, pva), accuracy_score(ys_val, pva > .5), int((m.named_steps["model"].coef_ != 0).sum())))
    return np.array(out)
sw_l2, sw_l1 = c_sweep("l2", C_l2), c_sweep("l1", C_l1)
bestC_l2, bestC_l1 = C_l2[sw_l2[:, 1].argmin()], C_l1[sw_l1[:, 1].argmin()]

fig, axes = plt.subplots(1, 3, figsize=(17, 4.2))
axes[0].plot(C_l2, sw_l2[:, 1], "o-", label="L2: val"); axes[0].plot(C_l2, sw_l2[:, 0], "o--", alpha=.5, label="L2: train")
axes[0].plot(C_l1, sw_l1[:, 1], "s-", label="L1: val"); axes[0].plot(C_l1, sw_l1[:, 0], "s--", alpha=.5, label="L1: train")
axes[0].set_xscale("log"); axes[0].set(title=f"Log-loss vs C (степень {HIGH_DEG})", xlabel="C (больше — слабее регуляризация)", ylabel="log-loss"); axes[0].legend()
axes[1].plot(C_l2, sw_l2[:, 2], "o-", label="L2"); axes[1].plot(C_l1, sw_l1[:, 2], "s-", label="L1")
axes[1].set_xscale("log"); axes[1].set(title="Val accuracy vs C", xlabel="C", ylabel="accuracy"); axes[1].legend()
axes[2].plot(C_l1, sw_l1[:, 3], "s-", color="tab:green"); axes[2].set_xscale("log")
axes[2].set(title="L1: число ненулевых весов (из 135)", xlabel="C", ylabel="ненулевых весов")
plt.show()
print(f"Лучшее C: L2 → {bestC_l2:.3g}, L1 → {bestC_l1:.3g}")

In [ ]:
models_syn = {
    "степень 1": poly_clf(1),
    f"степень {best_deg} (лучшая по log-loss)": poly_clf(best_deg),
    f"степень {HIGH_DEG}, C=1000 (слабая рег.)": poly_clf(HIGH_DEG, C=1000),
    f"степень {HIGH_DEG}, L2 C={bestC_l2:.2g}": poly_clf(HIGH_DEG, C=bestC_l2),
    f"степень {HIGH_DEG}, L1 C={bestC_l1:.2g}": poly_clf(HIGH_DEG, C=bestC_l1, penalty="l1"),
}
for m in models_syn.values():
    m.fit(Xs_tr, ys_tr)
names = list(models_syn)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, k in zip(axes, names[2:]):
    cs = plot_boundary(models_syn[k], Xs_tr, ys_tr, ax, k)
fig.colorbar(cs, ax=axes, label="P(класс 1)", shrink=.8); plt.show()

*Комментарий.* На одной и той же избыточной степени граница при слабой регуляризации «дрожит» и вырастает в шумовые острова, L2 её сглаживает, а L1 строит более простую границу из малого числа членов. Сравнение значений метрик — в таблице артефактов ниже.

## 3.3 Артефакты полученной модели

Для всех моделей — метрики; для лучшей по val log-loss — **веса, матрица ошибок, ROC-кривая, проверка формулы логистической регрессии вручную**.

In [ ]:
rows = []
for k, m in models_syn.items():
    ptr, pva = m.predict_proba(Xs_tr)[:, 1], m.predict_proba(Xs_val)[:, 1]
    mv = clf_metrics(ys_val, pva)
    rows.append({"модель": k, "признаков": m.named_steps["model"].coef_.size, "ненулевых": int((m.named_steps["model"].coef_ != 0).sum()),
                 "train acc": accuracy_score(ys_tr, ptr > .5), "val acc": mv["Accuracy"], "val F1": mv["F1"],
                 "val ROC-AUC": mv["ROC-AUC"], "val LogLoss": mv["LogLoss"]})
syn_tbl = pd.DataFrame(rows).set_index("модель")
display(syn_tbl.round(3))
best_syn_name = syn_tbl["val LogLoss"].idxmin()
best_syn = models_syn[best_syn_name]
print("Лучшая по val log-loss:", best_syn_name)

In [ ]:
# --- веса моделей степени 15 (стандартизованные полиномиальные признаки), symlog-шкала
fig, ax = plt.subplots(figsize=(13, 3.8))
width = .27; idx = np.arange(1, 136)
for i, (k, col, lab) in enumerate(zip(names[2:], ["tab:red", "tab:green", "tab:purple"], ["C=1000", "L2", "L1"])):
    ax.bar(idx + (i - 1) * width, models_syn[k].named_steps["model"].coef_.ravel(), width, color=col, label=lab)
ax.set_yscale("symlog", linthresh=.1); ax.set(title="Веса при полиномиальных признаках (степень 15, symlog)", xlabel="номер признака", ylabel="вес"); ax.legend()
plt.show()

# --- матрица ошибок и ROC лучшей модели на val
pva = best_syn.predict_proba(Xs_val)[:, 1]
fig, axes = plt.subplots(1, 2, figsize=(11, 4.3))
ConfusionMatrixDisplay.from_predictions(ys_val, pva > .5, ax=axes[0], colorbar=False, cmap="Blues"); axes[0].set_title("Матрица ошибок (val)"); axes[0].grid(False)
for k, m in list(models_syn.items())[:1] + [(best_syn_name, best_syn)]:
    RocCurveDisplay.from_predictions(ys_val, m.predict_proba(Xs_val)[:, 1], ax=axes[1], name=k)
axes[1].plot([0, 1], [0, 1], "k--", lw=1); axes[1].set_title("ROC-кривая (val)")
plt.show()

# --- проверка: predict_proba == sigmoid(w·x + b)
Z = best_syn[:-1].transform(Xs_val); lr = best_syn.named_steps["model"]
z_manual = Z @ lr.coef_.ravel() + lr.intercept_[0]
print(f"intercept_ = {lr.intercept_[0]:.4f}; число весов = {lr.coef_.size}")
print(f"max |predict_proba - sigmoid(w·x+b)| = {np.max(np.abs(best_syn.predict_proba(Xs_val)[:, 1] - sigmoid(z_manual))):.2e}  (≈0 → формула совпадает)")
print(f"max |decision_function - (w·x+b)|     = {np.max(np.abs(best_syn.decision_function(Xs_val) - z_manual)):.2e}")

*Комментарий.* Совпадение `predict_proba` с $\sigma(w^\top x+b)$ подтверждает, что вероятность — это сигмоида от линейной комбинации (полиномиальных) признаков, а граница `p=0.5` — линия уровня $w^\top x+b=0$ в пространстве признаков. Веса без регуляризации заметно крупнее и разнознаковы (компенсируют друг друга), у L1 большая часть весов равна нулю.

## 3.4 Выводы по синтетическим данным

In [ ]:
r = syn_tbl
l1row, l2row, hirow, d1row = r.iloc[4], r.iloc[3], r.iloc[2], r.iloc[0]
print(f'1. Линейной модели НЕ хватило: степень 1 → val accuracy = {d1row["val acc"]:.3f}, log-loss = {d1row["val LogLoss"]:.3f}.')
print(f'2. Хватает полинома степени ≈ {best_deg}: val log-loss минимален ({deg_df.val_logloss.min():.3f}), val accuracy = {deg_df.loc[deg_df.degree == best_deg, "val_acc"].iloc[0]:.3f}.')
print(f'3. Избыточная степень {HIGH_DEG} без сильной регуляризации: train acc = {hirow["train acc"]:.3f}, val acc = {hirow["val acc"]:.3f}, val log-loss = {hirow["val LogLoss"]:.3f} против {deg_df.val_logloss.min():.3f} у степени {best_deg}; разрыв train/val accuracy = {hirow["train acc"] - hirow["val acc"]:.3f} → признаки переобучения.')
print(f'4. Регуляризация при степени {HIGH_DEG}: L2 (C={bestC_l2:.2g}) val log-loss = {l2row["val LogLoss"]:.3f}; L1 (C={bestC_l1:.2g}) = {l1row["val LogLoss"]:.3f}, оставлено {int(l1row["ненулевых"])} из {int(l1row["признаков"])} весов.')
print('5. Accuracy слабо чувствительна к переобучению, log-loss — сильно: для вероятностных моделей смотрим на обе метрики.')

**Итог по части I.** Сложность модели (степень полинома) и сила регуляризации (`C`) — две ручки одного регулятора «смещение–дисперсия»: можно брать признаки «с запасом» и ограничивать их через `C`. L2 сглаживает границу, L1 дополнительно отбирает признаки.

# Часть II. Реальные данные: Adult Census Income (4.1–4.5)

## 4.1 Выбор набора данных

**Датасет:** *Adult* (Census Income, UCI / OpenML, `fetch_openml("adult", version=2)`) — данные переписи США 1994 г.

- **Задача бинарной классификации:** предсказать, превышает ли годовой доход человека **50 000 $** (`>50K`) или нет (`<=50K`).
- **Размер:** ≈48 842 строки (требовалось ≥ 1000), 14 признаков (требовалось ≥ 5): 5 числовых, 8 категориальных (+ технический вес `fnlwgt`).
- Есть **пропуски** в категориальных признаках и **дисбаланс классов** (≈24 % положительных) — реалистичные проблемы, на которых виден смысл предобработки и выбора метрик.
- Набор открытый (OpenML), загружается одной строкой, ручное скачивание не нужно.

**Этическое замечание.** `race` и `sex` — чувствительные атрибуты. Я **не использую их как признаки модели**; `sex` используется только на этапе анализа ошибок (проверка различий в качестве между группами).

In [ ]:
adult = fetch_openml("adult", version=2, as_frame=True)
df = adult.frame.copy()
TARGET = "class"
y_all = df[TARGET].astype(str).str.strip().str.startswith(">50K").astype(int).rename("income_gt_50K")
df = df.drop(columns=[TARGET])
obj_cols = df.select_dtypes(include=["category", "object"]).columns
df[obj_cols] = df[obj_cols].astype(object).replace("?", np.nan)

NUM = ["age", "education-num", "capital-gain", "capital-loss", "hours-per-week"]
CAT = ["workclass", "marital-status", "occupation", "relationship", "native-country"]
SENSITIVE = ["sex", "race"]            # в модель не подаём
X_all = df[NUM + CAT + SENSITIVE]

print("Размер:", X_all.shape, "| доля класса >50K:", f"{y_all.mean():.3f}")
display(X_all.head())
display(X_all[NUM].describe().T.round(2))
miss = X_all.isna().mean().mul(100).round(2); print("Пропуски, %:\n" + miss[miss > 0].to_string())

### Разведочный анализ (EDA)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.2))
cnt = y_all.value_counts().sort_index()
axes[0].bar(["<=50K", ">50K"], cnt.values, color=["tab:blue", "tab:red"])
for i, v in enumerate(cnt.values): axes[0].text(i, v, f"{v}\n({v/len(y_all):.1%})", ha="center", va="bottom")
axes[0].set(title="Баланс классов", ylabel="число людей"); axes[0].set_ylim(0, cnt.max() * 1.18)

sns.kdeplot(data=X_all.assign(y=y_all), x="age", hue="y", common_norm=False, fill=True, ax=axes[1], palette=["tab:blue", "tab:red"])
axes[1].set(title="Возраст по классам (0: <=50K, 1: >50K)")
tmp = X_all.assign(y=y_all)
rate = tmp.groupby(pd.cut(tmp["education-num"], [0, 8, 9, 10, 12, 13, 14, 16]), observed=True)["y"].mean()
axes[2].bar(range(len(rate)), rate.values, color="tab:purple"); axes[2].set_xticks(range(len(rate))); axes[2].set_xticklabels(["≤8", "9", "10", "11–12", "13", "14", "15–16"])
axes[2].set(title="Доля >50K по образованию", xlabel="education-num", ylabel="доля >50K")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.2))
tmp = X_all.assign(y=y_all)
for ax, col in zip(axes[:2], ["capital-gain", "capital-loss"]):
    nz = (tmp[col] > 0).mean()
    ax.hist(np.log1p(tmp[col][tmp[col] > 0]), bins=40, color="tab:orange", alpha=.85, edgecolor="white")
    ax.set(title=f"log(1+{col}) среди ненулевых; у {1-nz:.0%} людей значение 0", xlabel=f"log1p({col})", ylabel="число людей")
ms = tmp.groupby("marital-status")["y"].agg(["mean", "size"]).query("size > 100").sort_values("mean")
axes[2].barh(ms.index, ms["mean"], color="tab:red", alpha=.8); axes[2].set(title="Доля >50K по семейному положению", xlabel="доля >50K")
plt.tight_layout(); plt.show()

*Комментарий.* (1) Дисбаланс ≈ 76/24 → accuracy «всегда `<=50K`» уже даёт ≈76 %, поэтому будем смотреть **ROC-AUC, PR-AUC, F1**. (2) `capital-gain/loss` — у большинства ноль, а у остальных очень тяжёлый хвост → нужны логарифм и бинарные флаги. (3) Связь с возрастом и образованием **нелинейна** (доход растёт, затем падает) → полиномы. (4) Семейное положение сильно связано с доходом → категориальные признаки важны. Пропуски есть только в категориальных признаках.

## 4.2 Предобработка данных

**Разбиение:** train/test = 80/20, `stratify=y`, `random_state=42`; тест не используется до финала. Выбор моделей — 5-fold стратифицированная CV на train.

| Шаг | Зачем |
|---|---|
| `SimpleImputer(most_frequent)` + `OneHotEncoder(min_frequency=0.01)` | пропуски в категориях; редкие категории (<1 %) схлопываются в одну — защита от разреженности и переобучения |
| `FeatureEngineer` (свой класс) | `log1p(capital-gain/loss)`, флаги `has_capital_gain/loss`, `is_overtime` (>40 ч), категориальная `age_group` |
| `StandardScaler` | равный масштаб признаков для регуляризации и сходимости градиентных методов |
| `PolynomialFeatures` | нелинейные эффекты и взаимодействия числовых признаков |
| `ColumnTransformer` | числовые и категориальные признаки обрабатываются по-разному |

In [ ]:
X_model = X_all.drop(columns=SENSITIVE)               # признаки модели (без race/sex)
X_train, X_test, y_train, y_test = train_test_split(X_model, y_all, test_size=.2, random_state=SEED, stratify=y_all)
sens_test = X_all.loc[X_test.index, SENSITIVE]        # чувствительные атрибуты — только для анализа ошибок
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
print(f"train: {X_train.shape}, test: {X_test.shape}; доля >50K: train={y_train.mean():.3f}, test={y_test.mean():.3f}")

class FeatureEngineer(TransformerMixin, BaseEstimator):
    # Добавляет признаки; не обучается на данных (stateless), но оформлен как transformer для Pipeline.
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X = X.copy()
        X["log_capital_gain"] = np.log1p(X["capital-gain"])
        X["log_capital_loss"] = np.log1p(X["capital-loss"])
        X["has_capital_gain"] = (X["capital-gain"] > 0).astype(int)
        X["has_capital_loss"] = (X["capital-loss"] > 0).astype(int)
        X["is_overtime"] = (X["hours-per-week"] > 40).astype(int)
        X["age_group"] = pd.cut(X["age"], [0, 25, 35, 50, 65, np.inf], labels=["<=25", "26-35", "36-50", "51-65", "65+"]).astype(object)
        return X

NUM_ENG = ["age", "education-num", "log_capital_gain", "log_capital_loss", "hours-per-week",
           "has_capital_gain", "has_capital_loss", "is_overtime"]
CAT_ENG = CAT + ["age_group"]

def build_model(clf, *, scale=True, cat=True, fe=False, degree=1):
    # Собирает пайплайн; флаги включают отдельные улучшения (для таблицы абляции).
    num_cols = NUM_ENG if fe else NUM
    num_steps = []
    if degree > 1:
        num_steps.append(("poly", PolynomialFeatures(degree, include_bias=False)))
    if scale:
        num_steps.append(("scale", StandardScaler()))
    transformers = [("num", Pipeline(num_steps) if num_steps else "passthrough", num_cols)]
    if cat:
        transformers.append(("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                                              ("ohe", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=.01, sparse_output=False))]),
                             CAT_ENG if fe else CAT))
    steps = ([("fe", FeatureEngineer())] if fe else []) + [("prep", ColumnTransformer(transformers)), ("model", clf)]
    return Pipeline(steps)

_t = build_model(LogisticRegression(max_iter=2000), fe=True, degree=2).fit(X_train, y_train)
print("Число признаков после предобработки (fe, degree=2):", _t.named_steps["prep"].get_feature_names_out().size)

## 4.3 Baseline: «модель без всего»

`LogisticRegression` только на **5 сырых числовых признаках**: без масштабирования, без категориальных признаков, без обработки хвостов и дисбаланса, порог 0.5.

In [ ]:
results, fitted = [], {}
SCORING = {"roc_auc": "roc_auc", "pr_auc": "average_precision", "f1": "f1", "acc": "accuracy"}

def evaluate(name, model):
    # CV на train (выбор модели) + обучение на всём train + оценка на test (отчёт)
    cvr = cross_validate(model, X_train, y_train, cv=cv, scoring=SCORING, n_jobs=-1)
    model.fit(X_train, y_train)
    t = clf_metrics(y_test, model.predict_proba(X_test)[:, 1])
    results.append({"Модель": name, "CV ROC-AUC": cvr["test_roc_auc"].mean(), "CV PR-AUC": cvr["test_pr_auc"].mean(),
                    "CV PR-AUC std": cvr["test_pr_auc"].std(), "CV F1": cvr["test_f1"].mean(),
                    "Test Acc": t["Accuracy"], "Test Prec": t["Precision"], "Test Rec": t["Recall"], "Test F1": t["F1"],
                    "Test ROC-AUC": t["ROC-AUC"], "Test PR-AUC": t["PR-AUC"], "Test LogLoss": t["LogLoss"]})
    fitted[name] = model
    return model

M0 = "M0 Baseline: 5 числовых признаков, без всего"
baseline = evaluate(M0, build_model(LogisticRegression(max_iter=3000), scale=False, cat=False))
trivial = max(y_test.mean(), 1 - y_test.mean())
print(f"Для сравнения: accuracy тривиального классификатора «всегда <=50K» на тесте = {trivial:.3f}")
display(pd.DataFrame(results).set_index("Модель").round(4))

### Артефакты baseline-модели

In [ ]:
lr0 = baseline.named_steps["model"]
print(f"intercept_ = {lr0.intercept_[0]:.4f}")
display(pd.DataFrame({"признак": NUM, "вес": lr0.coef_.ravel()}).set_index("признак").T.round(4))

# веса на стандартизованных признаках — чтобы сравнивать «силу» признаков
std_lr = Pipeline([("sc", StandardScaler()), ("m", LogisticRegression(max_iter=3000))]).fit(X_train[NUM], y_train)
w_std = pd.Series(std_lr.named_steps["m"].coef_.ravel(), index=NUM).sort_values()

p0 = baseline.predict_proba(X_test)[:, 1]
fig, axes = plt.subplots(1, 4, figsize=(20, 4.4))
axes[0].barh(w_std.index, w_std.values, color=["tab:red" if v < 0 else "tab:blue" for v in w_std.values])
axes[0].set(title="Baseline: веса (стандартизованные признаки)\nlog-odds на +1σ признака", xlabel="вес")
ConfusionMatrixDisplay.from_predictions(y_test, p0 > .5, ax=axes[1], colorbar=False, cmap="Blues", display_labels=["<=50K", ">50K"]); axes[1].set_title("Матрица ошибок (порог 0.5)"); axes[1].grid(False)
RocCurveDisplay.from_predictions(y_test, p0, ax=axes[2], name="baseline"); axes[2].plot([0, 1], [0, 1], "k--", lw=1); axes[2].set_title("ROC-кривая")
PrecisionRecallDisplay.from_predictions(y_test, p0, ax=axes[3], name="baseline"); axes[3].axhline(y_test.mean(), color="k", ls="--", lw=1, label="случайный"); axes[3].legend(); axes[3].set_title("Precision–Recall кривая")
plt.tight_layout(); plt.show()

*Комментарий.* Веса логичны: сильнее всего прогноз двигают образование и возраст, затем часы работы и капитал. По матрице ошибок видно, что baseline **пропускает заметную часть высокодоходных** (низкий recall класса `>50K`), а accuracy выглядит «прилично» лишь из-за дисбаланса (сравните с тривиальным классификатором). Поэтому основные метрики дальше — ROC-AUC, PR-AUC и F1.

## 4.4 Улучшение качества модели

Каждый шаг добавляет **одну** идею (таблица абляции):

| Шаг | Что добавляем |
|---|---|
| M1 | стандартизация числовых признаков |
| M2 | + категориальные признаки (impute + one-hot, редкие категории схлопываются) |
| M3 | + свои признаки (`log1p` капитала, флаги, `is_overtime`, `age_group`) |
| M4 | + полиномы степени 2 для числовых признаков |
| M5 | + `class_weight="balanced"` (борьба с дисбалансом) |

In [ ]:
M1 = "M1 + StandardScaler"
M2 = "M2 + категориальные (impute + one-hot)"
M3 = "M3 + свои признаки (FeatureEngineer)"
M4 = "M4 + полиномы deg=2 (C=0.1)"
M5 = "M5 + class_weight=balanced"
LR = lambda **kw: LogisticRegression(max_iter=5000, **kw)

evaluate(M1, build_model(LR(), cat=False))
evaluate(M2, build_model(LR()))
evaluate(M3, build_model(LR(), fe=True))
evaluate(M4, build_model(LR(C=.1), fe=True, degree=2))
evaluate(M5, build_model(LR(C=.1, class_weight="balanced"), fe=True, degree=2))
display(pd.DataFrame(results).set_index("Модель").round(4))

### Подбор гиперпараметров

`GridSearchCV` (5-fold, критерий — **PR-AUC**): степень полинома ∈ {1, 2}, `C` ∈ {0.01, 0.1, 1, 10}, `class_weight` ∈ {None, balanced}.

In [ ]:
grid = GridSearchCV(build_model(LR(), fe=True, degree=2),
                    {"prep__num__poly__degree": [1, 2], "model__C": [.01, .1, 1, 10], "model__class_weight": [None, "balanced"]},
                    scoring="average_precision", cv=cv, n_jobs=-1).fit(X_train, y_train)
g = pd.DataFrame(grid.cv_results_)
g["степень"] = g["param_prep__num__poly__degree"].astype(int); g["C"] = g["param_model__C"].astype(float)
g["веса классов"] = g["param_model__class_weight"].map(lambda v: "balanced" if v == "balanced" else "None")
fig, axes = plt.subplots(1, 2, figsize=(12, 3.2), sharey=True)
for ax, (cw, sub) in zip(axes, g.groupby("веса классов")):
    sns.heatmap(sub.pivot(index="степень", columns="C", values="mean_test_score"), annot=True, fmt=".4f", cmap="viridis", ax=ax, cbar=False)
    ax.set_title(f"CV PR-AUC, class_weight={cw}")
plt.tight_layout(); plt.show()
bp = grid.best_params_
print("Лучшие параметры:", bp, f"| CV PR-AUC = {grid.best_score_:.4f}")

### Другие модели и регуляризация: L1 и `SGDClassifier`

- **L1-логистическая регрессия** (на линейных признаках, степень 1) — отбирает признаки; считаем число обнулённых весов;
- **`SGDClassifier(loss="log_loss")`** — та же логистическая регрессия, обучаемая стохастическим градиентным спуском (как в демонстрационном блокноте), с elastic-net регуляризацией.

In [ ]:
deg, Cb, cw = int(bp["prep__num__poly__degree"]), float(bp["model__C"]), bp["model__class_weight"]
M6 = f"M6 Подобранная LogReg: deg={deg}, C={Cb:g}, cw={cw}"
evaluate(M6, build_model(LR(C=Cb, class_weight=cw), fe=True, degree=deg))

# L1 считаем на линейных признаках (degree=1): на полиномах liblinear очень медленный, а для отбора признаков линейный набор нагляднее
l1_grid = GridSearchCV(build_model(LogisticRegression(penalty="l1", solver="liblinear", tol=1e-3, max_iter=500, class_weight=cw), fe=True, degree=1),
                       {"model__C": [.01, .03, .1, .3, 1]}, scoring="average_precision", cv=cv, n_jobs=-1).fit(X_train, y_train)
C1 = float(l1_grid.best_params_["model__C"])
M7 = f"M7 L1-регуляризация (линейные признаки): C={C1:g}"
evaluate(M7, build_model(LogisticRegression(penalty="l1", solver="liblinear", tol=1e-3, max_iter=500, class_weight=cw, C=C1), fe=True, degree=1))
w1 = fitted[M7].named_steps["model"].coef_.ravel()
print(f"L1: обнулено {int((w1 == 0).sum())} из {w1.size} весов")

M8 = "M8 SGDClassifier (log_loss, elasticnet)"
evaluate(M8, build_model(SGDClassifier(loss="log_loss", penalty="elasticnet", alpha=1e-4, l1_ratio=.15, class_weight=cw,
                                       max_iter=2000, tol=1e-4, random_state=SEED), fe=True, degree=deg))
res_df = pd.DataFrame(results).set_index("Модель")
display(res_df.round(4)); res_df.round(5).to_csv("results_summary.csv")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.8))
res_df[["CV PR-AUC", "Test PR-AUC"]].plot.barh(ax=ax, width=.75, color=["tab:blue", "tab:orange"])
ax.invert_yaxis(); lo, hi = res_df[["CV PR-AUC", "Test PR-AUC"]].min().min(), res_df[["CV PR-AUC", "Test PR-AUC"]].max().max()
ax.set_xlim(lo - .03, hi + .015); ax.set(title="Сравнение моделей по PR-AUC (больше — лучше)", xlabel="PR-AUC", ylabel="")
plt.tight_layout(); plt.show()

*Комментарий.* Критерий выбора — **CV PR-AUC** (чувствительна к редкому классу, не зависит от порога); Test PR-AUC — независимая проверка. Близость CV и Test говорит об отсутствии подгонки под тест. Заметьте: `class_weight="balanced"` почти не меняет ROC/PR-AUC (ранжирование), но **сдвигает вероятности** → растёт recall/F1 при пороге 0.5, но ухудшается log-loss (вероятности перестают быть калиброванными). Эффект порога разберём ниже.

## 4.5 Финальная модель, анализ ошибок и выводы

Финальная модель выбирается по максимальному CV PR-AUC. Затем **порог классификации** подбирается по out-of-fold-прогнозам на train (максимум F1) — так тест остаётся честным — и только после этого один раз оценивается на тесте.

In [ ]:
final_name = res_df["CV PR-AUC"].idxmax()
final = fitted[final_name]
print("Финальная модель:", final_name)

oof = cross_val_predict(final, X_train, y_train, cv=cv, method="predict_proba", n_jobs=-1)[:, 1]
ths = np.linspace(.05, .95, 91)
f1s = [f1_score(y_train, oof >= t) for t in ths]
thr = float(ths[int(np.argmax(f1s))])
print(f"Порог по OOF-F1 на train: {thr:.2f}  (F1 OOF = {max(f1s):.4f}; при 0.5 = {f1_score(y_train, oof >= .5):.4f})")

pf = final.predict_proba(X_test)[:, 1]
cmp_df = pd.DataFrame({"Baseline (порог 0.5)": clf_metrics(y_test, p0), "Финал (порог 0.5)": clf_metrics(y_test, pf),
                       f"Финал (порог {thr:.2f})": clf_metrics(y_test, pf, thr)}).T
display(cmp_df.round(4))

fig, ax = plt.subplots(figsize=(7, 3.6))
ax.plot(ths, f1s, color="tab:green"); ax.axvline(thr, color="k", ls=":", label=f"оптимум {thr:.2f}"); ax.axvline(.5, color="gray", ls="--", label="0.5")
ax.set(title="F1 на out-of-fold прогнозах train vs порог", xlabel="порог", ylabel="F1"); ax.legend(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(21, 4.6))
ConfusionMatrixDisplay.from_predictions(y_test, p0 > .5, ax=axes[0], colorbar=False, cmap="Blues", display_labels=["<=50K", ">50K"]); axes[0].set_title("Baseline, порог 0.5"); axes[0].grid(False)
ConfusionMatrixDisplay.from_predictions(y_test, pf >= thr, ax=axes[1], colorbar=False, cmap="Greens", display_labels=["<=50K", ">50K"]); axes[1].set_title(f"Финал, порог {thr:.2f}"); axes[1].grid(False)
RocCurveDisplay.from_predictions(y_test, p0, ax=axes[2], name="baseline"); RocCurveDisplay.from_predictions(y_test, pf, ax=axes[2], name="финал")
axes[2].plot([0, 1], [0, 1], "k--", lw=1); axes[2].set_title("ROC-кривые (test)")
PrecisionRecallDisplay.from_predictions(y_test, p0, ax=axes[3], name="baseline"); PrecisionRecallDisplay.from_predictions(y_test, pf, ax=axes[3], name="финал")
axes[3].axhline(y_test.mean(), color="k", ls="--", lw=1); axes[3].set_title("PR-кривые (test)")
plt.tight_layout(); plt.show()

*Комментарий.* Финальная модель выше baseline по всей ROC- и PR-кривой, а не только в одной точке; подбор порога по OOF переносит часть ошибок из «пропущенных высокодоходных» в «ложные тревоги» — выбор порога зависит от цены ошибок, а не от метрики как таковой.

In [ ]:
# --- калибровка и распределение вероятностей
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for nm, p in [("baseline", p0), ("финал", pf)]:
    fr, mp = calibration_curve(y_test, p, n_bins=10, strategy="quantile"); axes[0].plot(mp, fr, "o-", label=nm)
axes[0].plot([0, 1], [0, 1], "k--", lw=1, label="идеал"); axes[0].set(title="Калибровочная кривая (test)", xlabel="средняя предсказанная вероятность", ylabel="доля >50K"); axes[0].legend()
for c, col in [(0, "tab:blue"), (1, "tab:red")]:
    axes[1].hist(pf[y_test.values == c], bins=40, alpha=.6, color=col, label=f"факт: {'<=50K' if c == 0 else '>50K'}", density=True)
axes[1].axvline(thr, color="k", ls=":", label=f"порог {thr:.2f}"); axes[1].set(title="Распределение P(>50K) по истинным классам", xlabel="предсказанная вероятность", ylabel="плотность"); axes[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
# --- где модель ошибается: по группам (возраст, пол, образование)
a = pd.DataFrame({"y": y_test.values, "pred": (pf >= thr).astype(int), "age_group": FeatureEngineer().transform(X_test)["age_group"].values,
                  "sex": sens_test["sex"].values, "edu": pd.cut(X_test["education-num"], [0, 9, 12, 16], labels=["до школы/HS", "колледж", "высшее"]).astype(str).values})
def grp(col):
    return a.groupby(col).apply(lambda d: pd.Series({"n": len(d), "доля >50K (факт)": d.y.mean(), "доля >50K (прогноз)": d.pred.mean(),
                                                     "recall": recall_score(d.y, d.pred, zero_division=0), "precision": precision_score(d.y, d.pred, zero_division=0)}), include_groups=False)
fig, axes = plt.subplots(1, 3, figsize=(17, 4.2))
for ax, col, order in [(axes[0], "age_group", ["<=25", "26-35", "36-50", "51-65", "65+"]), (axes[1], "edu", ["до школы/HS", "колледж", "высшее"]), (axes[2], "sex", None)]:
    t = grp(col); t = t.loc[order] if order else t
    t[["recall", "precision"]].plot.bar(ax=ax, rot=0, color=["tab:red", "tab:blue"]); ax.set(title=f"Recall и precision по группам: {col}", xlabel="", ylim=(0, 1))
plt.tight_layout(); plt.show()
display(pd.concat({"пол": grp("sex"), "возраст": grp("age_group")}).round(3))

*Комментарий.* Качество **неоднородно по группам**: в группах с малой долей высокодоходных (как правило, молодые и с низким образованием) положительных примеров мало, и это сказывается на recall/precision; у разных групп различаются и доля предсказанных `>50K`, и precision/recall. Такие различия отражают структуру данных (1994 г.), и пользоваться моделью для реальных решений о людях без аудита нельзя — это важное ограничение.

In [ ]:
# --- самые влиятельные признаки финальной модели
est = final
names_out = [n.replace("num__", "").replace("cat__", "").replace("ohe__", "") for n in est.named_steps["prep"].get_feature_names_out()]
w = pd.Series(est.named_steps["model"].coef_.ravel(), index=names_out)
top = w.reindex(w.abs().sort_values(ascending=False).index[:16]).sort_values()
fig, ax = plt.subplots(figsize=(8, 5.4))
ax.barh(top.index, top.values, color=["tab:red" if v < 0 else "tab:blue" for v in top.values])
ax.set(title="Финальная модель: 16 самых больших по модулю весов\n(log-odds; числовые признаки стандартизованы)", xlabel="вес"); plt.tight_layout(); plt.show()
print(f"intercept_ = {est.named_steps['model'].intercept_[0]:.4f}; всего весов: {w.size}")

joblib.dump({"model": final, "threshold": thr}, "adult_income_model.joblib")
loaded = joblib.load("adult_income_model.joblib")
print("Загруженная модель даёт те же прогнозы:", bool(np.allclose(loaded["model"].predict_proba(X_test)[:, 1], pf)))

### Выводы

In [ ]:
b = res_df.loc[M0]; f = res_df.loc[final_name]; ft = clf_metrics(y_test, pf, thr); fh = clf_metrics(y_test, pf)
print(f'1. БАЗА. Baseline: accuracy = {b["Test Acc"]:.3f} (тривиальный классификатор: {trivial:.3f}), recall класса >50K = {b["Test Rec"]:.3f}, F1 = {b["Test F1"]:.3f}, ROC-AUC = {b["Test ROC-AUC"]:.3f}, PR-AUC = {b["Test PR-AUC"]:.3f}.')
print(f'2. ФИНАЛ ({final_name}): ROC-AUC = {f["Test ROC-AUC"]:.3f}, PR-AUC = {f["Test PR-AUC"]:.3f}; при пороге 0.5 F1 = {fh["F1"]:.3f}, при подобранном пороге {thr:.2f}: F1 = {ft["F1"]:.3f}, recall = {ft["Recall"]:.3f}, precision = {ft["Precision"]:.3f}, accuracy = {ft["Accuracy"]:.3f}.')
print(f'   Прирост PR-AUC: {f["Test PR-AUC"] - b["Test PR-AUC"]:+.3f}; ROC-AUC: {f["Test ROC-AUC"] - b["Test ROC-AUC"]:+.3f}; F1 (порог {thr:.2f} vs baseline): {ft["F1"] - b["Test F1"]:+.3f}.')
print(f'3. ПЕРЕОБУЧЕНИЕ. CV PR-AUC = {f["CV PR-AUC"]:.3f} ± {f["CV PR-AUC std"]:.3f} против Test PR-AUC = {f["Test PR-AUC"]:.3f}: {"согласованы → переобучения нет" if abs(f["CV PR-AUC"] - f["Test PR-AUC"]) < 0.02 else "расхождение заметно → стоит усилить регуляризацию"}.')
print('4. АБЛЯЦИЯ (ΔCV PR-AUC при добавлении очередной идеи):')
_s = [M0, M1, M2, M3, M4, M5]
for _a, _b in zip(_s[:-1], _s[1:]):
    _d = res_df.loc[_b, "CV PR-AUC"] - res_df.loc[_a, "CV PR-AUC"]; print(f'   {_b}: {_d:+.4f} ({"лучше" if _d > 0 else "не лучше"})')
print(f'5. КАЛИБРОВКА. Log-loss: baseline {b["Test LogLoss"]:.3f}, финал {f["Test LogLoss"]:.3f}.')

**Содержательные выводы**

- **Метрики адекватны, если выбраны под задачу.** При дисбаланса 76/24 accuracy почти не отличается от тривиального классификатора, поэтому качество оценивалось ROC-AUC, PR-AUC и F1; порог выбирался по out-of-fold-прогнозам, тест использовался один раз.
- **Что дало прирост.** Вклад каждой идеи (масштабирование, категориальные признаки, свои признаки, полиномы, веса классов) виден в таблице абляции и в автоматическом выводе выше (ΔCV PR-AUC); не каждая идея обязана улучшать метрику, поэтому финальная модель выбирается по CV, а не «по ощущению». `class_weight` и порог **не улучшают ранжирование** (ROC/PR-AUC почти не меняются), а меняют компромисс precision/recall — это решение о цене ошибок, а не «улучшение модели».
- **Как распределены ошибки.** Ошибки сосредоточены там, где вероятность близка к порогу (см. гистограмму вероятностей); качество и доля предсказанных `>50K` различаются между возрастными, образовательными и половыми группами (см. графики и таблицу выше) — без аудита по группам использовать модель нельзя.
- **Калибровка.** Взвешивание классов делает вероятности смещёнными (log-loss и калибровочная кривая хуже); если нужны честные вероятности — использовать модель без весов и подбирать порог, либо калибровать (`CalibratedClassifierCV`).
- **Ограничения и что дальше.** Линейная модель не ловит сложные взаимодействия (например, образование × профессия × возраст); следующий шаг — градиентный бустинг/деревья как верхняя планка, калибровка вероятностей и fairness-аудит.